# Generating MV grids with pylovo

pylovo generates the LV grids of a version (`pylovo-generate`); its MV stage connects them to the HV/MV
substations. The MV stage works on a stored LV version and never changes its LV grids:

- **Nodes.** One MV node per LV grid (the MV/LV station, greenfield or brownfield), one per MV customer (buildings
  above `MV_DIRECT_CONNECTION_LOAD_THRESHOLD_KW` of the LV version; small ones hang on the busbar of the nearest
  station).
- **Substations and supply areas.** HV/MV substations from OSM, matched to their eGon-data substation (the 110 kV
  bus) where the eGon-data table is available. Every node is supplied by the substation nearest to it along the
  streets. One substation and its nodes are one MV grid (district).
- **Routing.** A persistent routing graph of the region's streets and field paths with settlement flags (urban and
  rural settlements from the buildings and OSM land use). MV lines mapped in OSM are preferred corridors.
- **Rings.** Open rings from the busbar (Clarke-Wright savings with local search), at most
  `MV_MAX_STATIONS_PER_RING` members (a rural member counts half), one normally-open point per ring. Remote rural
  stations hang on stubs, large customers get dedicated feeders, and lines along the mapped OSM MV lines are overhead
  lines. Normally-open links join rings inside a district, and normally-open ties join neighbouring districts.
- **Voltage level.** 20 kV or 10 kV for all districts, or `auto` (`MV_NOMINAL_VOLTAGE_KV`): 10 kV in districts
  whose installed MV/LV capacity lies mostly in dense municipalities (`mv_10kv_min_density_mva_km2`), else 20 kV.
- **Electrical checks.** pandapower load case and the two (n-1) cases with a repair loop, HV/MV transformer sizing,
  and a voltage budget per station: MV/LV tap and LV busbar voltage at the LV design point.

This notebook shows how to run the stage and lists the MV variants of a version (read-only). The results are
analysed in `1_mv_results_overview.ipynb` to `4_mv_lv_coupling.ipynb`.

**Prerequisites**

- pylovo with the `notebooks` extra (`uv sync --extra notebooks`) and this notebook running on the kernel of the
  project environment (`.venv`), see `notebook_tutorials/README.md`.
- A `.env` in the repository root that points to your pylovo database and a completed `pylovo-setup` (it records
  the migration `0009_mv_grids` of the MV tables).
- A generated LV version (`pylovo-generate`) and the OSM power layer of the InfDB (`INFDB_OSM_SCHEMA` in the
  `.env`). The eGon-data substations (`pylovo` or the opendata schema) are optional: without them the sites keep no
  eGon-data match.

In [ ]:
# Parameters
version_id = None  # LV version of the MV results; None: VERSION_ID of config/config_generation_lv.yaml
dbname = None  # expected database name of the .env; None: no check
repo_root = None   # repository root; None: two levels above this notebook

## Setup

The first cell changes to the repository root, so that pylovo reads the `.env` and `config/` of this checkout
(and writes its log to `log/`), and checks the database name when `dbname` is set. `mv_notebook_utils.py` next to
this notebook holds the shared helpers of the MV notebooks.

In [ ]:
import argparse
import os
import sys
from dataclasses import fields
from pathlib import Path

if repo_root is None:
    here = Path.cwd().resolve()
    repo_root = next((p for p in (here, *here.parents)
                      if (p / "pyproject.toml").is_file() and (p / "src" / "pylovo").is_dir()), None)
    if repo_root is None:
        raise RuntimeError("Set repo_root to the MV checkout before running this notebook.")
repo_root = Path(repo_root).expanduser().resolve()

# A kernel registered for another checkout can silently import its pylovo installation.
import importlib.util

spec = importlib.util.find_spec("pylovo")
expected = repo_root / "src" / "pylovo" / "__init__.py"
if spec is None or spec.origin is None or Path(spec.origin).resolve() != expected:
    raise RuntimeError(
        f"Select the notebook kernel at {repo_root / '.venv' / 'bin' / 'python'}, "
        "restart it, then Run All. The current kernel imports pylovo from "
        f"{None if spec is None else spec.origin!r}."
    )
os.chdir(repo_root)
sys.path.insert(0, str(repo_root / "notebook_tutorials" / "mv_grids"))

from pylovo import config_loader

if dbname is not None:
    assert config_loader.DBNAME == dbname, f"the .env points to {config_loader.DBNAME!r}, not {dbname!r}"

import pandas as pd
from IPython.display import Markdown, display

import mv_notebook_utils as mvu
from pylovo.mv_grid import options
from pylovo.mv_grid.model import MvRunSummary
from pylovo.mv_grid.parameters import MvParameters

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

dbc = mvu.read_only_client()
version_id, variants, listing = mvu.resolve(dbc, version_id, select="current")
default = mvu.default_variant(listing)
print(f"Database {config_loader.DBNAME}, LV version {version_id}, done MV variants: {', '.join(variants) or 'none'}, "
      f"default variant: {default or 'none'}")

## Running the MV stage

**After the LV grids (generate hook).** With `MV_GRID_GENERATION: True` in `config/config_generation_mv.yaml`, or
with `--mv`, `pylovo-generate` runs the MV stage once after all PLZ of the call, for the configured MV variant
(`MV_VARIANT_ID`) of `VERSION_ID`. The LV grids are committed per PLZ first, so an MV failure never undoes them;
`--no-mv` switches the stage off for one call.

```bash
uv run pylovo-generate --plz <PLZ> <PLZ> --mv
```

**On a stored LV version (`pylovo-mv generate`).** The usual way: any LV version, any MV variant, parameters on
the command line.

```bash
# all PLZ with LV grids of the version, shipped parameters, four district workers
uv run pylovo-mv generate --version <version> --all --variant base --comment "shipped parameters" --jobs 4

# the PLZ of some municipalities (AGS) only, rings of at most 15 members
uv run pylovo-mv generate --version <version> --ags <AGS> <AGS> --variant ring15 \
    --set MV_MAX_STATIONS_PER_RING=15
```

- `--plz`, `--ags` or `--all` select the PLZ; the planning region is the union of their PLZ polygons.
- `--variant` and `--comment` name the MV variant (default `MV_VARIANT_ID`, `MV_VARIANT_COMMENT`).
- `--set KEY=VALUE` changes an `MV_*` key of `config/config_generation_mv.yaml` for this variant (see below);
  every changed key needs a new variant id.
- `--jobs` sets the district workers (default `min(4, N_JOBS)`, capped by the free memory), `--no-parallel` plans
  the districts in one process.
- A second run of the same variant **resumes**: done districts are skipped, pending and failed ones are planned
  again. A larger PLZ set or a changed routing graph rebuilds all districts; `--replace` deletes the variant first.

**Other commands.**

```bash
uv run pylovo-mv list --version <version>                                # status, PLZ, districts, km, stale
uv run pylovo-mv couple --version <version> --variant base --jobs 2       # LV design-point check, merged MV+LV flows
uv run pylovo-mv export --version <version> --variant base --grid <id> --output grid.json --case n1a
uv run pylovo-mv export --version <version> --variant base --grid <id> --output merged.json --coupled
uv run pylovo-mv graph --version <version> --all --qa                    # build or reuse the routing graph
uv run pylovo-mv boundary --version <version> --variant base --output boundary.csv  # HV boundary: eTraGo bus, P, Q
uv run pylovo-mv display --version <version> --variant base               # store the display geometry again
uv run pylovo-mv delete --version <version> --variant ring15             # one variant (all without --variant)
```

`couple` stores its results in `mv_node.coupled_*` and `mv_grid.metrics['coupling']` (`4_mv_lv_coupling.ipynb`).
`export` writes the stored pandapower net of one district (`--case` load, n1a or n1b), or with `--coupled` the
merged MV+LV net. `boundary` writes the HV boundary of a variant (one row per supplying HV/MV substation with its
eTraGo 110 kV bus, load-case P and Q, transformers and their loading) to a CSV file. `display` stores the display
geometry (lanes where lines share a route, busbar circles) of variants built before it existed again. `list` names
the MV stage version of a variant planned with an older one: it stays readable, but it cannot be resumed or
coupled. `pylovo-delete networks` also deletes the MV variants that
contain the deleted PLZ. Every output line starts with ✓ or ✗; the exit code is 0 only when the variant is done.

The help of `pylovo-mv` and its subcommands (read from the parser, nothing runs):

In [ ]:
from pylovo.cli.mv import build_parser

parser = build_parser()
print(parser.format_help())
commands = next(a for a in parser._actions if isinstance(a, argparse._SubParsersAction))
for name in ("generate", "couple", "export"):
    print(f"--- pylovo-mv {name} ---")
    print(commands.choices[name].format_help())

## MV parameters and `--set`

The generation configuration has two files:

- `config/config_generation_lv.yaml`: the LV stage (`VERSION_ID`, transformer placement, cables, ...);
- `config/config_generation_mv.yaml`: the 9 `MV_*` keys of the MV stage.

The tables below show the `MV_*` keys (value of this configuration and shipped default) and the constants of the
stage. Both are stored per MV variant in `mv_variant.mv_parameters` (`snapshot path`), not in the snapshot of the
LV version. `MV_GRID_GENERATION` is an execution flag only.

`pylovo-mv generate --set KEY=VALUE` takes the `MV_*` keys except `MV_GRID_GENERATION`, `MV_VARIANT_ID` and
`MV_VARIANT_COMMENT` (use `pylovo-generate --mv`, `--variant` and `--comment`). The constants are changed in Python
with `MvParameters.replace` (field names) and passed to `run_mv_stage`:

```python
from pylovo.mv_grid.parameters import MvParameters
from pylovo.mv_grid.stage import run_mv_stage

params = MvParameters.from_config().replace(variant_id="ls0", comment="no local search", local_search_rounds=0)
summary = run_mv_stage("<version>", None, params, jobs=2)   # None: every PLZ with LV grids of the version
```

The default conductor types are 240 mm² cable and 122-AL1 overhead (with voltage-specific standard types).

In [ ]:
config = MvParameters.from_config()
keys = pd.DataFrame([{
    "key": s.name, "this configuration": getattr(config, s.field) if s.field else getattr(config_loader, s.name),
    "default": s.default, "choices": ", ".join(map(str, s.choices)), "snapshot path": ".".join(s.path),
    "source": s.source} for s in options.YAML_KEYS.values()]).set_index("key")
display(keys)
constants = pd.DataFrame([{
    "constant": s.name, "default": s.default, "group": s.group, "snapshot path": ".".join(s.path),
    "source": s.source} for s in options.CONSTANTS.values()]).set_index("constant")
display(constants)
print(f"MV stage version {options.MV_STAGE_VERSION}; not settable with --set: {', '.join(options.NOT_SETTABLE)}")

## MV variants

An MV variant is identified by its LV version and a variant id of at most 10 characters, and is defined by its
parameters. Each variant keeps its own parameter snapshot. Examples of variants and the question they answer:

| Id | Change | Question |
|---|---|---|
| `base` | none | the shipped parameters: load and (n-1) planning |
| `base` on a second LV version | none | station set: greenfield against brownfield (OSM/LoD2) stations |
| `ring15` | `--set MV_MAX_STATIONS_PER_RING=15` | ring size |
| `util50` | `--set MV_LINE_PLANNING_UTILIZATION=0.5` | planning utilization |
| `kv10` | `--set MV_NOMINAL_VOLTAGE_KV=10` (or `20`, `auto`) | nominal voltage |
| `drop8` | `--set MV_MAX_VOLTAGE_DROP_PERCENT=8` | voltage drop in normal operation |
| `cable` | `--set MV_STANDARD_CABLE=<pandapower type> MV_OVERHEAD_LINE=<pandapower type>` | conductor types (default: the most used German MV cable and overhead line) |
| `ls0` | `MvParameters.replace(local_search_rounds=0)` | value of the local search |
| `osm1` | `MvParameters.replace(osm_line_cost_factor=1.0)` | OSM MV lines as preferred corridors |

The variants of this LV version (`pylovo-mv list`), then the parameters in which its done variants of the current
MV stage version differ and their run summaries:

In [ ]:
columns = ["mv_variant_id", "mv_variant_comment", "status", "mv_stage_version", "n_plz", "n_grids", "n_done", "n_failed",
           "n_stations", "circuit_km", "cable_share", "stale", "graph_id", "created_at", "finished_at"]
display(mvu.pretty(listing[columns].set_index("mv_variant_id")))
rows = {name: dbc.variant_row(version_id, name) for name in variants}
dbc.conn.rollback()
table = pd.DataFrame({n: mvu.variant_parameters(r["mv_parameters"]) for n, r in rows.items()})
if len(table.columns) > 1:
    table = table[table.astype(str).nunique(axis=1) > 1]
display(table)
names = {f.name for f in fields(MvRunSummary)}
for name, row in rows.items():
    summary = row.get("summary") or {}
    try:
        print(MvRunSummary(**{k: v for k, v in summary.items() if k in names}).describe())
    except TypeError:
        print(f"{name}: {summary}")
    if summary.get("ties"):
        print(f"  ties: {summary['ties']}")

## Routing graph

The routing graph is reference data shared by the variants of a region (`pylovo.mv_graph*`, identified by the
municipalities, the build parameters and a fingerprint of the source tables). Its QA tables: size, km per street
class, connected components, vertex degrees and settlements.

In [ ]:
for graph_id in sorted(set(listing["graph_id"].dropna())):
    display(Markdown(f"**Graph {graph_id}**"))
    for name, frame in dbc.graph_qa(graph_id).items():
        display(Markdown(f"*{name}*"))
        display(frame.drop(columns=["ags"], errors="ignore").head(15))
    dbc.conn.rollback()

## Reading the results in Python

`DatabaseClient` reads the results of a variant without the `mv_grid.net` column: `read_mv_layers` (GeoDataFrames
of nodes, lines, substations, districts and settlements for the maps of `pylovo.plotting.mv_grid`),
`read_mv_rings`, `read_variant_stats` (the view `pylovo.mv_variant_stats`) and `read_mv_grid_net` (the stored
pandapower net of one district with its load-case results). The example reads the default variant: `MV_VARIANT_ID`
of the configuration if the version has it, else the newest done variant of the current MV stage version.

In [ ]:
stats = dbc.read_variant_stats(version_id)
dbc.conn.rollback()
display(mvu.pretty(stats[stats["mv_variant_id"].isin(variants)].drop(columns=["n_generators"], errors="ignore")
                   .set_index("mv_variant_id")))
if default:
    layers = dbc.read_mv_layers(version_id, default)
    print({name: len(frame) for name, frame in layers.items()})
    done = layers["grids"][layers["grids"]["status"] == "done"]
    if len(done):
        net = dbc.read_mv_grid_net(int(done["mv_grid_id"].iloc[0]))
        dbc.conn.rollback()
        print(net)